# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Get dataset metadata (using to_json for preview)
metadata = dataset.metadata.to_json()
print(f"{metadata.get('name')}: {metadata.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by Croissant schema.

In [ ]:
# Display record sets and their fields (using @id)
print('Record Sets (@id):')
for record_set in dataset.record_sets:
    print(f"  - RecordSet @id: {record_set.id}")
    print(f"    Name: {record_set.name}")
    print('    Fields:')
    for field in record_set.fields:
        print(f"      - Field @id: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All operations reference record sets and fields by their `@id`.

In [ ]:
# Extract all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
print('Available Record Sets (@id):', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Preview columns in each DataFrame
for rs_id, df in dataframes.items():
    print(f'Columns for RecordSet {rs_id}:', df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. Fields are referenced by their Croissant `@id`.

In [ ]:
# Choose record set and numeric field for demonstration (replace with actual @id from above overview)
## For illustration, we'll take the first RecordSet and the first FLOAT/INTEGER field we find.

record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in dataset.record_sets:
    for field in rs.fields:
        if (field.data_type or '').lower() in ['float', 'integer', 'number']:
            record_set_id = rs.id
            numeric_field_id = field.id
            # Find any groupable (categorical or text) field
            for f in rs.fields:
                if f.id != numeric_field_id and (f.data_type or '').lower() in ['text', 'string', '']:
                    group_field_id = f.id
                    break
            break
    if record_set_id:
        break

if not (record_set_id and numeric_field_id and record_set_id in dataframes):
    print('No suitable record set and numeric field found for EDA.')
else:
    df = dataframes[record_set_id]

    # Ensure field ID is in dataframe columns
    if numeric_field_id in df.columns:
        threshold = df[numeric_field_id].quantile(0.75)  # Example threshold at 75th percentile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records from {record_set_id} where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping by a categorical field if available
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print(f'Field {numeric_field_id} not found in columns of {record_set_id}.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (record_set_id and numeric_field_id and record_set_id in dataframes and numeric_field_id in dataframes[record_set_id].columns):
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id is available, show boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} in {record_set_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The `mlcroissant` library makes it easy to load structured, FAIR datasets defined by Croissant schemas.
- The FAIR\⁲ dataset includes record sets with fields such as protein accession, description, and quantitative measurements as described in the Croissant schema.
- Exploratory Data Analysis (EDA) can be performed by referencing fields and record sets via their Croissant `@id`, enabling programmatic and reproducible workflows.
- Further analyses can be scripted depending on downstream questions regarding protein abundance, group differences, or modifications.